<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Automatización profesional de Excel con `openpyxl`**

## *Del Excel manual al workbook reproducible*

En este laboratorio entraremos en el segundo gran bloque del curso: automatización de ficheros Excel.

Trabajaremos con libros reales `.xlsx` generados por el propio notebook.

La metodología será:

> **crear workbook → inspeccionar → detectar inconsistencias → leer → escribir → calcular → formatear → validar → guardar → reabrir → interpretar**

No nos limitaremos a escribir datos. Automatizaremos:

- libros;
- hojas;
- celdas;
- fórmulas;
- estilos;
- anchos;
- formatos numéricos;
- validación básica;
- múltiples hojas;
- resúmenes;
- comprobaciones de integridad.

## Objetivos

Al finalizar deberías poder:

1. Crear un workbook con `openpyxl`.
2. Crear, renombrar y eliminar hojas.
3. Leer celdas individuales.
4. Escribir valores.
5. Recorrer filas y columnas.
6. Importar datos tabulares.
7. Crear fórmulas.
8. Aplicar estilos.
9. Aplicar formatos numéricos.
10. Ajustar anchos de columna.
11. Congelar paneles.
12. Detectar valores ausentes o inconsistentes.
13. Corregir datos programáticamente.
14. Crear hojas de resumen.
15. Guardar y reabrir un libro.
16. Validar que la automatización ha generado el resultado esperado.

## 1. Preparación

In [ ]:
from pathlib import Path
import shutil
import statistics

import pandas as pd
import matplotlib.pyplot as plt

try:
    import openpyxl
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import (
        Font,
        PatternFill,
        Alignment,
        Border,
        Side
    )
    from openpyxl.utils import get_column_letter
    OPENPYXL_DISPONIBLE = True
except ImportError:
    OPENPYXL_DISPONIBLE = False

print(
    "openpyxl disponible:",
    OPENPYXL_DISPONIBLE
)

if OPENPYXL_DISPONIBLE:
    print(
        "Versión:",
        openpyxl.__version__
    )

# 2. Crear entorno de trabajo

In [ ]:
raiz = Path(
    "demo_excel_openpyxl"
)

if raiz.exists():
    shutil.rmtree(
        raiz
    )

raiz.mkdir()

ruta_excel = (
    raiz
    / "ventas_demo.xlsx"
)

print(
    "Ruta:",
    ruta_excel.resolve()
)

# 3. Crear primer workbook

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb = Workbook()

    ws = wb.active

    print(
        "Hoja inicial:",
        ws.title
    )

# 3.1 Renombrar hoja

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws.title = "Ventas"

    print(
        wb.sheetnames
    )

# 3.2 Crear hojas nuevas

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb.create_sheet(
        "Clientes"
    )

    wb.create_sheet(
        "Resumen"
    )

    print(
        wb.sheetnames
    )

### Interpretación

Un workbook puede contener múltiples hojas y cada una puede asumir una responsabilidad distinta.

Ejemplo:

```text
Ventas     → datos operativos
Clientes   → maestros
Resumen    → indicadores
```

# 4. Escribir cabeceras

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_ventas = wb[
        "Ventas"
    ]

    cabeceras = [
        "ID",
        "Cliente",
        "Region",
        "Producto",
        "Unidades",
        "Precio",
        "Descuento",
        "Importe"
    ]

    for columna, valor in enumerate(
        cabeceras,
        start=1
    ):
        ws_ventas.cell(
            row=1,
            column=columna,
            value=valor
        )

    [
        celda.value
        for celda
        in ws_ventas[1]
    ]

# 5. Crear datos de prueba

In [ ]:
ventas = [
    [1, "Cliente A", "Norte", "Producto X", 2, 100.0, 0.10],
    [2, "Cliente B", "Sur", "Producto Y", 5, 40.0, 0.00],
    [3, "Cliente C", "Norte", "Producto X", 1, 100.0, 0.05],
    [4, "Cliente D", "Este", "Producto Z", 4, 75.0, 0.20],
    [5, "Cliente E", "Sur", "Producto Y", 3, 40.0, None],
    [6, "Cliente F", "Oeste", "Producto X", -1, 100.0, 0.10],
    [7, "Cliente G", "Norte", "Producto Z", 2, 75.0, 1.20],
]

pd.DataFrame(
    ventas,
    columns=[
        "ID",
        "Cliente",
        "Region",
        "Producto",
        "Unidades",
        "Precio",
        "Descuento"
    ]
)

### Inconsistencias deliberadas

Antes de escribir, ya podemos observar:

- descuento ausente;
- unidades negativas;
- descuento superior al 100 %.

No corregimos todavía.

# 6. Escribir filas

In [ ]:
if OPENPYXL_DISPONIBLE:

    for fila_idx, fila in enumerate(
        ventas,
        start=2
    ):

        for columna_idx, valor in enumerate(
            fila,
            start=1
        ):
            ws_ventas.cell(
                row=fila_idx,
                column=columna_idx,
                value=valor
            )

    print(
        "Filas escritas:",
        ws_ventas.max_row
    )

# 7. Añadir fórmulas de importe

In [ ]:
if OPENPYXL_DISPONIBLE:

    for fila in range(
        2,
        ws_ventas.max_row + 1
    ):

        ws_ventas.cell(
            row=fila,
            column=8,
            value=(
                f"=E{fila}*F{fila}*(1-G{fila})"
            )
        )

    print(
        ws_ventas[
            "H2"
        ].value
    )

### Interpretación

La fórmula queda escrita en Excel.

`openpyxl` no calcula fórmulas como Excel; guarda la expresión para que Excel la evalúe al abrir el fichero.

# 8. Guardar primera versión

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb.save(
        ruta_excel
    )

    print(
        "Guardado:",
        ruta_excel.exists()
    )

# 9. Reabrir workbook

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb2 = load_workbook(
        ruta_excel
    )

    print(
        wb2.sheetnames
    )

# 10. Leer celdas

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws2 = wb2[
        "Ventas"
    ]

    print(
        "A1:",
        ws2[
            "A1"
        ].value
    )

    print(
        "B2:",
        ws2[
            "B2"
        ].value
    )

    print(
        "H2:",
        ws2[
            "H2"
        ].value
    )

# 11. Recorrer filas

In [ ]:
if OPENPYXL_DISPONIBLE:

    for fila in ws2.iter_rows(
        min_row=1,
        max_row=4,
        values_only=True
    ):
        print(
            fila
        )

# 12. Convertir hoja a DataFrame

In [ ]:
if OPENPYXL_DISPONIBLE:

    valores = list(
        ws2.iter_rows(
            values_only=True
        )
    )

    cabecera = valores[0]

    datos = valores[1:]

    df_ventas = pd.DataFrame(
        datos,
        columns=cabecera
    )

    df_ventas

### Interpretación

Podemos combinar:

- `openpyxl` para manipulación del workbook;
- `pandas` para análisis tabular.

No son herramientas excluyentes.

# 13. Detectar inconsistencias

In [ ]:
if OPENPYXL_DISPONIBLE:

    inconsistencias = pd.DataFrame({
        "fila_excel":
            range(
                2,
                len(
                    df_ventas
                ) + 2
            ),

        "unidades_invalidas":
            (
                df_ventas[
                    "Unidades"
                ] <= 0
            ),

        "descuento_ausente":
            df_ventas[
                "Descuento"
            ].isna(),

        "descuento_fuera_rango":
            (
                (
                    df_ventas[
                        "Descuento"
                    ] < 0
                )
                |
                (
                    df_ventas[
                        "Descuento"
                    ] > 1
                )
            ).fillna(
                False
            )
    })

    inconsistencias

# 13.1 Ver solo filas problemáticas

In [ ]:
if OPENPYXL_DISPONIBLE:

    problemas = inconsistencias[
        inconsistencias[
            [
                "unidades_invalidas",
                "descuento_ausente",
                "descuento_fuera_rango"
            ]
        ].any(
            axis=1
        )
    ]

    problemas

# 13.2 Visualizar problemas

In [ ]:
if OPENPYXL_DISPONIBLE:

    conteo_problemas = {
        "Unidades invalidas":
            int(
                inconsistencias[
                    "unidades_invalidas"
                ].sum()
            ),

        "Descuento ausente":
            int(
                inconsistencias[
                    "descuento_ausente"
                ].sum()
            ),

        "Descuento fuera rango":
            int(
                inconsistencias[
                    "descuento_fuera_rango"
                ].sum()
            )
    }

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        conteo_problemas.keys(),
        conteo_problemas.values()
    )

    plt.ylabel(
        "Número de filas"
    )

    plt.title(
        "Inconsistencias detectadas en Excel"
    )

    plt.xticks(
        rotation=20
    )

    plt.tight_layout()
    plt.show()

### Interpretación

Ahora sí sabemos exactamente qué errores debemos corregir.

La corrección será explícita y auditable.

# 14. Definir reglas de corrección

Reglas didácticas:

- descuento ausente → 0;
- unidades <= 0 → enviar a revisión y convertir a 1 solo para demo;
- descuento > 1 → limitar a 1.

En un proyecto real estas reglas deberían venir de negocio.

# 15. Aplicar correcciones en workbook

In [ ]:
if OPENPYXL_DISPONIBLE:

    for fila_excel in range(
        2,
        ws2.max_row + 1
    ):

        unidades = ws2.cell(
            row=fila_excel,
            column=5
        ).value

        descuento = ws2.cell(
            row=fila_excel,
            column=7
        ).value

        if unidades is not None and unidades <= 0:
            ws2.cell(
                row=fila_excel,
                column=5,
                value=1
            )

        if descuento is None:
            ws2.cell(
                row=fila_excel,
                column=7,
                value=0
            )

        elif descuento > 1:
            ws2.cell(
                row=fila_excel,
                column=7,
                value=1
            )

    print(
        "Correcciones aplicadas."
    )

# 16. Validar correcciones

In [ ]:
if OPENPYXL_DISPONIBLE:

    valores_corregidos = list(
        ws2.iter_rows(
            values_only=True
        )
    )

    df_corregido = pd.DataFrame(
        valores_corregidos[1:],
        columns=valores_corregidos[0]
    )

    print(
        "Unidades invalidas:",
        int(
            (
                df_corregido[
                    "Unidades"
                ] <= 0
            ).sum()
        )
    )

    print(
        "Descuento ausente:",
        int(
            df_corregido[
                "Descuento"
            ].isna().sum()
        )
    )

    print(
        "Descuento fuera rango:",
        int(
            (
                (
                    df_corregido[
                        "Descuento"
                    ] < 0
                )
                |
                (
                    df_corregido[
                        "Descuento"
                    ] > 1
                )
            ).sum()
        )
    )

# 17. Aplicar estilo a cabeceras

In [ ]:
if OPENPYXL_DISPONIBLE:

    relleno = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    fuente = Font(
        bold=True,
        color="FFFFFF"
    )

    alineacion = Alignment(
        horizontal="center"
    )

    for celda in ws2[1]:
        celda.fill = relleno
        celda.font = fuente
        celda.alignment = alineacion

    print(
        "Cabecera formateada."
    )

# 18. Aplicar bordes

In [ ]:
if OPENPYXL_DISPONIBLE:

    borde_fino = Side(
        style="thin"
    )

    borde = Border(
        left=borde_fino,
        right=borde_fino,
        top=borde_fino,
        bottom=borde_fino
    )

    for fila in ws2.iter_rows(
        min_row=1,
        max_row=ws2.max_row,
        min_col=1,
        max_col=8
    ):
        for celda in fila:
            celda.border = borde

    print(
        "Bordes aplicados."
    )

# 19. Formatos numéricos

In [ ]:
if OPENPYXL_DISPONIBLE:

    for fila in range(
        2,
        ws2.max_row + 1
    ):

        ws2.cell(
            row=fila,
            column=6
        ).number_format = '#,##0.00 €'

        ws2.cell(
            row=fila,
            column=7
        ).number_format = '0%'

        ws2.cell(
            row=fila,
            column=8
        ).number_format = '#,##0.00 €'

    print(
        "Formatos numéricos aplicados."
    )

# 20. Ajustar anchos automáticamente

In [ ]:
if OPENPYXL_DISPONIBLE:

    for columna in ws2.columns:

        letra = get_column_letter(
            columna[0].column
        )

        maximo = 0

        for celda in columna:

            valor = (
                ""
                if celda.value is None
                else str(
                    celda.value
                )
            )

            maximo = max(
                maximo,
                len(
                    valor
                )
            )

        ws2.column_dimensions[
            letra
        ].width = (
            min(
                maximo + 2,
                35
            )
        )

    print(
        "Anchos ajustados."
    )

# 21. Congelar cabecera

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws2.freeze_panes = "A2"

    print(
        "Freeze panes:",
        ws2.freeze_panes
    )

# 22. Crear hoja de clientes

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_clientes = wb2[
        "Clientes"
    ]

    clientes = sorted(
        set(
            df_corregido[
                "Cliente"
            ]
        )
    )

    ws_clientes.append([
        "Cliente"
    ])

    for cliente in clientes:
        ws_clientes.append([
            cliente
        ])

    print(
        "Clientes:",
        len(
            clientes
        )
    )

# 23. Estilizar hoja Clientes

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_clientes[
        "A1"
    ].font = Font(
        bold=True
    )

    ws_clientes[
        "A1"
    ].fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )

    ws_clientes.column_dimensions[
        "A"
    ].width = 25

# 24. Crear resumen en Python

In [ ]:
if OPENPYXL_DISPONIBLE:

    df_resumen = (
        df_corregido
        .groupby(
            "Region",
            as_index=False
        )
        .agg(
            Operaciones=(
                "ID",
                "count"
            ),
            Unidades=(
                "Unidades",
                "sum"
            ),
            Precio_Medio=(
                "Precio",
                "mean"
            )
        )
    )

    df_resumen

# 24.1 Visualizar resumen

In [ ]:
if OPENPYXL_DISPONIBLE:

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        df_resumen[
            "Region"
        ],
        df_resumen[
            "Unidades"
        ]
    )

    plt.ylabel(
        "Unidades"
    )

    plt.title(
        "Unidades por región"
    )

    plt.tight_layout()
    plt.show()

# 25. Escribir hoja Resumen

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_resumen = wb2[
        "Resumen"
    ]

    ws_resumen.delete_rows(
        1,
        ws_resumen.max_row
    )

    ws_resumen.append(
        list(
            df_resumen.columns
        )
    )

    for fila in df_resumen.itertuples(
        index=False,
        name=None
    ):
        ws_resumen.append(
            list(
                fila
            )
        )

    print(
        "Resumen escrito."
    )

# 26. Estilizar resumen

In [ ]:
if OPENPYXL_DISPONIBLE:

    for celda in ws_resumen[1]:
        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )

        celda.fill = PatternFill(
            fill_type="solid",
            fgColor="548235"
        )

        celda.alignment = Alignment(
            horizontal="center"
        )

    for columna in range(
        1,
        ws_resumen.max_column + 1
    ):
        ws_resumen.column_dimensions[
            get_column_letter(
                columna
            )
        ].width = 18

    print(
        "Resumen estilizado."
    )

# 27. Añadir indicadores globales

In [ ]:
if OPENPYXL_DISPONIBLE:

    fila_inicio = (
        ws_resumen.max_row
        + 3
    )

    ws_resumen.cell(
        row=fila_inicio,
        column=1,
        value="Indicador"
    )

    ws_resumen.cell(
        row=fila_inicio,
        column=2,
        value="Valor"
    )

    indicadores = [
        (
            "Número de ventas",
            len(
                df_corregido
            )
        ),
        (
            "Unidades totales",
            int(
                df_corregido[
                    "Unidades"
                ].sum()
            )
        ),
        (
            "Clientes únicos",
            int(
                df_corregido[
                    "Cliente"
                ].nunique()
            )
        ),
        (
            "Precio medio",
            float(
                df_corregido[
                    "Precio"
                ].mean()
            )
        )
    ]

    for offset, (
        nombre,
        valor
    ) in enumerate(
        indicadores,
        start=1
    ):
        ws_resumen.cell(
            row=fila_inicio + offset,
            column=1,
            value=nombre
        )

        ws_resumen.cell(
            row=fila_inicio + offset,
            column=2,
            value=valor
        )

    print(
        "Indicadores añadidos."
    )

# 28. Guardar versión corregida

In [ ]:
if OPENPYXL_DISPONIBLE:

    ruta_final = (
        raiz
        / "ventas_demo_corregido.xlsx"
    )

    wb2.save(
        ruta_final
    )

    print(
        "Guardado:",
        ruta_final.exists()
    )

# 29. Reabrir para validar persistencia

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb_final = load_workbook(
        ruta_final,
        data_only=False
    )

    print(
        wb_final.sheetnames
    )

    print(
        "Freeze:",
        wb_final[
            "Ventas"
        ].freeze_panes
    )

    print(
        "Formula H2:",
        wb_final[
            "Ventas"
        ][
            "H2"
        ].value
    )

# 30. Validar estructura del workbook

In [ ]:
if OPENPYXL_DISPONIBLE:

    assert set(
        wb_final.sheetnames
    ) == {
        "Ventas",
        "Clientes",
        "Resumen"
    }

    assert (
        wb_final[
            "Ventas"
        ].max_row
        == len(
            df_corregido
        ) + 1
    )

    assert (
        wb_final[
            "Ventas"
        ][
            "H2"
        ].value
        is not None
    )

    print(
        "Workbook validado."
    )

# 31. Crear función reusable de estilo

In [ ]:
if OPENPYXL_DISPONIBLE:

    def estilizar_cabecera(
        worksheet,
        fila=1,
        color="1F4E78"
    ):
        relleno = PatternFill(
            fill_type="solid",
            fgColor=color
        )

        fuente = Font(
            bold=True,
            color="FFFFFF"
        )

        for celda in worksheet[
            fila
        ]:
            celda.fill = relleno
            celda.font = fuente
            celda.alignment = Alignment(
                horizontal="center"
            )

        return worksheet

    print(
        "Función creada."
    )

# 32. Crear función para ajustar columnas

In [ ]:
if OPENPYXL_DISPONIBLE:

    def ajustar_columnas(
        worksheet,
        max_width=35
    ):
        for columna in worksheet.columns:

            letra = get_column_letter(
                columna[
                    0
                ].column
            )

            longitud = max(
                len(
                    str(
                        celda.value
                    )
                )
                if celda.value
                is not None
                else 0
                for celda
                in columna
            )

            worksheet.column_dimensions[
                letra
            ].width = min(
                longitud + 2,
                max_width
            )

        return worksheet

# 33. Crear función para exportar DataFrame a hoja

In [ ]:
if OPENPYXL_DISPONIBLE:

    def dataframe_a_hoja(
        dataframe,
        worksheet,
        limpiar=True
    ):
        if limpiar:
            worksheet.delete_rows(
                1,
                worksheet.max_row
            )

        worksheet.append(
            list(
                dataframe.columns
            )
        )

        for fila in dataframe.itertuples(
            index=False,
            name=None
        ):
            worksheet.append(
                list(
                    fila
                )
            )

        return worksheet

### Interpretación

Empezamos a construir una pequeña librería interna de utilidades para automatizar Excel.

# 34. Provocar un nombre de hoja inválido

In [ ]:
if OPENPYXL_DISPONIBLE:

    try:
        wb_final.create_sheet(
            "Ventas/2026"
        )

    except ValueError as error:
        print(
            type(error).__name__,
            "->",
            error
        )

### Interpretación

Los nombres de hoja tienen restricciones.

Un generador de informes debe validar o normalizar nombres antes de crear hojas.

# 35. Normalizar nombre de hoja

In [ ]:
if OPENPYXL_DISPONIBLE:

    def nombre_hoja_seguro(
        nombre,
        max_len=31
    ):
        prohibidos = (
            ":",
            "\\",
            "/",
            "?",
            "*",
            "[",
            "]"
        )

        limpio = str(
            nombre
        )

        for caracter in prohibidos:
            limpio = limpio.replace(
                caracter,
                "_"
            )

        limpio = (
            limpio[:max_len]
            .strip()
        )

        if not limpio:
            limpio = (
                "Hoja"
            )

        return limpio


    print(
        nombre_hoja_seguro(
            "Ventas/2026"
        )
    )

# 36. Crear hoja con nombre seguro

In [ ]:
if OPENPYXL_DISPONIBLE:

    nombre_seguro = (
        nombre_hoja_seguro(
            "Ventas/2026"
        )
    )

    if nombre_seguro not in wb_final.sheetnames:
        wb_final.create_sheet(
            nombre_seguro
        )

    print(
        wb_final.sheetnames
    )

# 37. Mini-práctica integrada

Construye automáticamente un Excel de ventas con al menos 100 filas.

Debe incluir:

1. hoja `Ventas`;
2. hoja `Clientes`;
3. hoja `Resumen`;
4. datos con inconsistencias deliberadas;
5. detección de NAs;
6. detección de unidades inválidas;
7. detección de descuentos fuera de rango;
8. corrección;
9. fórmulas;
10. estilos;
11. formatos numéricos;
12. anchos;
13. freeze panes;
14. resumen por región;
15. indicadores globales;
16. guardado y reapertura;
17. validaciones automáticas.

# 38. Checklist de automatización Excel

- [ ] ¿El workbook existe?
- [ ] ¿Las hojas esperadas existen?
- [ ] ¿Las cabeceras son correctas?
- [ ] ¿Se han detectado inconsistencias antes de corregir?
- [ ] ¿Las reglas de corrección están documentadas?
- [ ] ¿Las fórmulas están escritas?
- [ ] ¿Los formatos numéricos son adecuados?
- [ ] ¿La cabecera es legible?
- [ ] ¿Los anchos son razonables?
- [ ] ¿Se ha congelado la fila de cabecera?
- [ ] ¿Existe una hoja resumen?
- [ ] ¿Se ha guardado?
- [ ] ¿Se ha reabierto?
- [ ] ¿Se ha validado persistencia?

# 39. Ejercicios finales

### Ejercicio 1 — Workbook
Crea un libro nuevo.

### Ejercicio 2 — Hojas
Crea `Datos`, `Resumen` y `Errores`.

### Ejercicio 3 — Escritura
Escribe una tabla manualmente.

### Ejercicio 4 — Lectura
Lee tres celdas.

### Ejercicio 5 — Iteración
Recorre filas con `iter_rows`.

### Ejercicio 6 — Fórmula
Añade una columna calculada.

### Ejercicio 7 — Inconsistencia
Introduce valores ausentes.

### Ejercicio 8 — Validación
Detecta datos fuera de rango.

### Ejercicio 9 — Corrección
Aplica reglas explícitas.

### Ejercicio 10 — Estilo
Formatea cabeceras.

### Ejercicio 11 — Número
Aplica porcentaje y moneda.

### Ejercicio 12 — Ancho
Ajusta columnas automáticamente.

### Ejercicio 13 — Freeze
Congela la primera fila.

### Ejercicio 14 — Resumen
Agrupa por una categoría.

### Ejercicio 15 — DataFrame
Exporta un DataFrame a hoja.

### Ejercicio 16 — Reapertura
Guarda y vuelve a cargar.

### Ejercicio 17 — Nombre de hoja
Provoca un nombre inválido y normalízalo.

### Ejercicio 18 — Reto
Genera un informe Excel completo y validado a partir de datos tabulares.

# 40. Preguntas de reflexión

1. ¿Qué aporta `openpyxl`?
2. ¿Cuándo usarías `pandas` y cuándo `openpyxl`?
3. ¿`openpyxl` calcula fórmulas?
4. ¿Por qué detectar inconsistencias antes de corregir?
5. ¿Qué ventajas tiene automatizar estilos?
6. ¿Por qué reabrir el fichero al final?
7. ¿Qué diferencia hay entre contenido y presentación?
8. ¿Por qué validar nombres de hojas?
9. ¿Qué ventajas tiene una hoja de resumen?
10. ¿Qué funciones reutilizables crearías para futuros informes?
11. ¿Qué parte del proceso podría parametrizarse?
12. ¿Cómo evolucionará esto hacia generación masiva de informes?

# 41. Cierre

Ya sabemos crear y manipular Excel de forma programática.

El siguiente laboratorio profundizará en la parte de presentación y reporting:

```text
openpyxl
   ↓
estilos
   ↓
gráficos
   ↓
tablas
   ↓
resúmenes
   ↓
informes Excel profesionales
```